# SymboLM (v2.1) — 5k Contrastive SFT Syntactic Seeding on GPU
**Author & Principal Architect:** Sanish Gyawali (Rupandehi, Nepal)  
**AI Systems Collaborator:** Antigravity (Google DeepMind)  
**Repository:** https://github.com/gyawalisanish0/symboLM  

### Objective:
Fine-tunes the base model on the expanded **5,000-sample SymboLM v2.1 Contrastive Curriculum** covering 12 invariant families (including linear algebra, kinematics, combinations, sequences, word problems, Olympiad invariants, atomic facts, and intent dialogue) with 30% counterfactual rollback traces (`checkpoint → ✗ → backtrack → replan: hyp [...] → ✓`). Runs for 3 epochs (~938 optimizer steps) to permanently align policy weights with the formal Tri-Register grammar.

In [ ]:
# Cell 1: Environment & GPU Accelerator Verification
import os
import sys
import torch

print(f'Python version: {sys.version}')
print(f'Torch version : {torch.__version__}')
if torch.cuda.is_available():
    print(f'[OK] NVIDIA CUDA detected! Found {torch.cuda.device_count()} GPU(s):')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB VRAM)')
else:
    print('[WARNING] CUDA is NOT available! Check Kaggle accelerator settings.')


In [ ]:
# Cell 2: Install required packages & dependencies
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q 'transformers>=4.48.0' 'peft>=0.14.0' 'accelerate>=1.2.0' 'bitsandbytes>=0.45.0' 'datasets>=3.2.0' 'sympy>=1.12'


In [ ]:
# Cell 3: Clone latest SymboLM repository from GitHub
!rm -rf /kaggle/working/symboLM
!git clone https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
%cd /kaggle/working/symboLM
!git status


In [ ]:
# Cell 4: Generate Verified 5,000-Sample Contrastive Curriculum
!python -m data.build_concept_math_dataset --count 5000
!python -m data.build_contrastive_curriculum
!sha256sum data/concept_math_contrastive_v2_1.jsonl


In [ ]:
# Cell 5: Launch SymboLM v2.1 3-Epoch SFT Training on GPU
# 5,000 samples x 3 epochs = 15,000 / 16 (batch 8 x grad accum 2) = ~938 optimizer steps
!python -m training.sft_train \
    --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --train_file data/concept_math_contrastive_v2_1.jsonl \
    --output_dir checkpoints/symbolm_v2_adapter \
    --epochs 3 \
    --batch_size 8 \
    --grad_accum 2 \
    --lr 2e-4 \
    --no_quant


In [ ]:
# Cell 6: Post-Training Policy Audit across Cognitive Depths & New Domains
import torch
from inference.generate import load_symbo_model
from symbolic.grammar_fsm import SymboLogitsProcessor

print('[AUDIT] Loading freshly fine-tuned SymboLM v2.1 Adapter...')
model, tokenizer = load_symbo_model('deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', 'checkpoints/symbolm_v2_adapter')
model.eval()

test_queries = [
    'What is the atomic number of Carbon?',
    'If x + 1/x = 6, find the exact value of x^2 + 1/x^2.',
    'Find the smallest positive integer n such that n ≡ 4 (mod 7) and n ≡ 9 (mod 11).',
    'Find the highest power of 3 that divides 200! (Legendre formula).',
    'In a right triangle with integer legs a and b, inradius is r = 6 and perimeter is P = 72. Find area.',
    'Solve for x: 3x - 5 = 16.',
    'From a group of 8 distinct candidates, how many ways can a committee of 3 members be chosen?',
    'A body moves with initial velocity 10 m/s and constant acceleration 4 m/s^2. What is its velocity after 5 seconds?'
]

for q in test_queries:
    print('\n' + '=' * 65)
    print(f'Query: {q}')
    prompt = f'<｜begin of sentence｜><｜User｜>{q}<｜Assistant｜><think>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    proc = SymboLogitsProcessor(tokenizer, prompt_len=inputs.input_ids.shape[1])
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.2,
            do_sample=True,
            logits_processor=[proc],
            pad_token_id=tokenizer.eos_token_id
        )
    gen_text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
    print('[Generation]')
    print(gen_text)


In [ ]:
# Cell 7: Package & Archive Trained v2.1 Policy Weights
import os
adapter_cfg = '/kaggle/working/symboLM/checkpoints/symbolm_v2_adapter/adapter_config.json'
if not os.path.exists(adapter_cfg):
    raise FileNotFoundError(f'CRITICAL: {adapter_cfg} not found! Training did not complete.')

!tar -czf /kaggle/working/symboLM_v2_contrastive_policy.tar.gz -C /kaggle/working/symboLM/checkpoints/symbolm_v2_adapter .
print('[COMPLETE] SymboLM v2.1 policy weights archived to /kaggle/working/symboLM_v2_contrastive_policy.tar.gz')
